# Latent AE CNN

Trains `TDSConvCTC` on **pre-computed autoencoder latent vectors** instead of raw EMG.
The spectrogram front-end is replaced by a single linear projection.

**Data:** `data/emg_latent_ae_v2.hdf5` — 27,971 frames × 1024-dim float32 @ 62.5 Hz (32 ms/frame)  
**Model:** `Linear(1024 → num_features) → ReLU → TDSConvEncoder → Linear → LogSoftmax`  
**Checkpoint:** `Playground_Ben/checkpoints/best_latent_cnn.pt`

```bash
source /home/benforbes/emg2qwerty/venv/bin/activate
cd ~/C247_mumbikaijonathanben && jupyter notebook
```

## 1. Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path
import torch

REPO_ROOT = Path(os.getcwd())
while not (REPO_ROOT / 'emg2qwerty').is_dir():
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

PLAYGROUND  = REPO_ROOT / 'Playground_Ben'
SCRIPT      = PLAYGROUND / 'scripts/train_latent_cnn.py'
CHECKPOINT  = PLAYGROUND / 'checkpoints/best_latent_cnn.pt'
HYPERPARAMS = PLAYGROUND / 'checkpoints/best_hyperparams_cnn_latent.yaml'
DATA        = REPO_ROOT  / 'data/emg_latent_ae_v2.hdf5'

print(f'Repo root : {REPO_ROOT}')
print(f'Data file : {DATA}  exists={DATA.exists()}')

## 2. (Optional) Hyperparameter Tuning

Two-phase Bayesian search (20 coarse + 10 fine trials). Takes ~1–2 hours on a GPU.
Skip if `best_hyperparams_cnn_latent.yaml` already exists.

In [ ]:
if HYPERPARAMS.exists():
    print(f'Hyperparams already exist: {HYPERPARAMS}  — skipping tuning.')
else:
    result = subprocess.run(
        [sys.executable, str(PLAYGROUND / 'scripts/hyperparam_tuner_cnn.py'),
         '--coarse-trials', '20', '--coarse-epochs', '10',
         '--fine-trials', '10',   '--fine-epochs', '20'],
        cwd=str(REPO_ROOT)
    )
    print(f'Tuner exit code: {result.returncode}')

## 3. Train

Uses tuned hyperparameters if available, otherwise falls back to defaults  
(`lr=3e-4`, `num_features=768`, `block_channels=24`, `num_blocks=4`, `kernel_width=32`).

In [ ]:
cmd = [sys.executable, str(SCRIPT), '--epochs', '150', '--batch-size', '32']
if HYPERPARAMS.exists():
    cmd += ['--from-hyperparams', str(HYPERPARAMS)]
    print('Using tuned hyperparameters.')
else:
    cmd += ['--lr', '3e-4', '--window-length', '500', '--stride', '50']
    print('Using default hyperparameters.')

print('Command:', ' '.join(cmd))
result = subprocess.run(cmd, cwd=str(REPO_ROOT))
print(f'Exit code: {result.returncode}')

## 4. Inspect Checkpoint

In [ ]:
if CHECKPOINT.exists():
    ckpt = torch.load(CHECKPOINT, map_location='cpu')
    print('Latent AE CNN — Best Checkpoint')
    print(f'  Epoch   : {ckpt["epoch"]}')
    print(f'  Val CER : {ckpt["val_cer"]:.2f}%')
    print(f'  File    : {CHECKPOINT}')
else:
    print('Checkpoint not found — run the training cell first.')

## 5. Architecture Summary

```
Input:  (T, N, 1024)  — latent frames at 62.5 Hz
  └─ Linear(1024 → num_features)  +  ReLU
  └─ TDSConvEncoder(num_features, block_channels, kernel_width)
  └─ Linear(num_features → num_classes)  +  LogSoftmax
Output: (T', N, 71)   — CTC log-probs
```

**Key differences from raw-EMG baseline:**
- No `SpectrogramNorm` or `MultiBandRotationInvariantMLP`
- Single linear projection replaces the entire spectrogram front-end
- Input at 62.5 Hz vs 2000 Hz → 32× fewer time steps
- Faster iteration: ~2.4 min/epoch vs ~10 min/epoch on same hardware